> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Filtering data using a CTE


### Business requirement: Extract successful transactions whose amounts exceed ₦30,000.

In [ ]:
-- The CTE filters successful payments; the outer query applies the amount threshold.
WITH
    successful_transactions AS (
        SELECT
            transaction_id,
            customer_id,
            amount,
            status
        FROM payment_transactions
        WHERE
            status = 'Success'
    )
SELECT
    transaction_id,
    customer_id,
    amount,
    status
FROM successful_transactions
WHERE
    amount > 30000
ORDER BY amount DESC;

## CTE with aggregation

### Business requirement: Calculate total successful spending per customer ID and identify those who have spent at least ₦50,000.

In [ ]:
-- Aggregate successful payments once per customer, then filter the grouped spending.
WITH
    customer_spending AS (
        SELECT
            customer_id,
            COUNT(*) AS transaction_count,
            SUM(amount) AS total_spent
        FROM payment_transactions
        WHERE
            status = 'Success'
        GROUP BY
            customer_id
    )
SELECT
    customer_id,
    transaction_count,
    total_spent
FROM customer_spending
WHERE
    total_spent >= 50000
ORDER BY total_spent DESC;

## CTE with JOIN

### Business requirement: Calculate successful spending per customer and enrich the results with customer information. Preserve every registered customer, including customers with zero successful transactions.

In [ ]:
-- Keep every registered customer; COALESCE converts missing spending and counts to zero.
WITH
    customer_spending AS (
        SELECT
            customer_id,
            COUNT(*) AS total_transactions,
            SUM(amount) AS total_spent
        FROM payment_transactions
        WHERE
            status = 'Success'
        GROUP BY
            customer_id
    )
SELECT
    c.customer_id,
    c.customer_name,
    c.country,
    COALESCE(s.total_transactions, 0) AS total_transactions,
    COALESCE(s.total_spent, 0) AS total_spent
FROM
    customers c
    LEFT JOIN customer_spending AS s ON c.customer_id = s.customer_id
ORDER BY s.total_spent DESC, c.customer_id ASC;

## Multiple dependent CTEs

### Business requirement: Classify registered customers based on their total successful spending.

In [ ]:
-- Chain CTEs with commas; classify registered customers, including those with no payments.
WITH successful_transactions AS (
    SELECT transaction_id, customer_id, amount
    FROM payment_transactions
    WHERE status = 'Success'
), customer_spending AS (
    SELECT customer_id, COUNT(*) AS total_transactions, SUM(amount) AS total_spent
    FROM successful_transactions
    GROUP BY customer_id
), customer_segment AS (
    SELECT c.customer_id, c.customer_name,
        COALESCE(s.total_transactions, 0) AS total_transactions,
        COALESCE(s.total_spent, 0) AS total_spent,
        CASE WHEN COALESCE(s.total_spent, 0) >= 100000 THEN 'Premium'
             WHEN COALESCE(s.total_spent, 0) >= 50000 THEN 'Standard'
             ELSE 'Basic' END AS customer_segment
    FROM customers c
    LEFT JOIN customer_spending s ON c.customer_id = s.customer_id
)
SELECT * FROM customer_segment
ORDER BY total_spent DESC, customer_id;